# Diabetes Risk Prediction: Comparative Analysis of Logistic Regression and Random Forest
### Academic Machine Learning Project (Course Outcomes CO1 - CO6)

**Author:** BTech Machine Learning Student  
**Dataset:** Pima Indians Diabetes Database (National Institute of Diabetes and Digestive and Kidney Diseases)

---

### Course Outcomes Alignment:
- **CO1:** End-to-end Machine Learning Lifecycle (Data -> Features -> Training -> Evaluation -> Serving)
- **CO2:** Logistic Regression with Feature Engineering and Regularization
- **CO3:** Decision Tree / Random Forest as Tree-Based Supervised Learning Models
- **CO4:** PCA-Based Dimensionality Reduction & Feature Analysis
- **CO5:** Stratified K-Fold Cross-Validation, Classification Metrics & Hyperparameter Optimization
- **CO6:** Basic ML Engineering Practices (Model Packaging & API Serving)

## 1. Environment Setup and Data Ingestion (CO1)
Loading required libraries for data manipulation, visualization, modeling, and evaluation.

In [ ]:
# CO1: End-to-end Machine Learning Lifecycle - Environment Setup
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)

sns.set_theme(style="whitegrid")
print("Environment initialized successfully.")

## 2. Data Inspection and Exploratory Analysis (CO1)
Loading authentic Pima Indians Diabetes dataset and inspecting distributions.

In [ ]:
# CO1: Data Loading & Inspection
df = pd.read_csv("../dataset/diabetes.csv")
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

### Physiological Zero Detection (Data Cleaning)
In Glucose, BloodPressure, SkinThickness, Insulin, and BMI, a zero value represents missing data rather than a true biological measurement.

In [ ]:
zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
for col in zero_cols:
    c = (df[col] == 0).sum()
    print(f"{col:15}: {c:3} zero values ({c / len(df) * 100:.1f}%)")

## 3. Data Cleaning & Leakage Prevention (CO1 & CO2)
We replace zero with NaN and split data 80/20 with stratification BEFORE fitting imputer and scaler.

In [ ]:
# CO1: Stratified Split & Leak-Free Imputation/Scaling
df_clean = df.copy()
for col in zero_cols:
    df_clean[col] = df_clean[col].replace(0, np.nan)

X = df_clean.drop("Outcome", axis=1)
y = df_clean["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

imputer = SimpleImputer(strategy="median").fit(X_train)
X_train_imp = pd.DataFrame(imputer.transform(X_train), columns=X.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

scaler = StandardScaler().fit(X_train_imp)
X_train_scl = pd.DataFrame(scaler.transform(X_train_imp), columns=X.columns)
X_test_scl = pd.DataFrame(scaler.transform(X_test_imp), columns=X.columns)

print(f"X_train shape: {X_train_scl.shape}, X_test shape: {X_test_scl.shape}")

## 4. PCA-Based Dimensionality Reduction & Feature Analysis (CO4)
Standardized features are projected into principal components to compute explained variance and visualize clinical distribution.

In [ ]:
# CO4: PCA-Based Dimensionality Reduction / Feature Analysis
pca = PCA(random_state=42)
pca.fit(X_train_scl)

exp_var = pca.explained_variance_ratio_
cum_var = np.cumsum(exp_var)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(1, 9), exp_var, label="Individual Explained Variance", alpha=0.7)
ax.step(range(1, 9), cum_var, where="mid", color="red", label="Cumulative Explained Variance")
ax.axhline(0.85, color="gray", linestyle="--", label="85% Threshold")
ax.set_xlabel("Principal Component")
ax.set_ylabel("Explained Variance Ratio")
ax.set_title("CO4: Scree Plot of Principal Components")
ax.legend()
plt.show()

## 5. Model 1: Logistic Regression with Regularization (CO2)
Tuning inverse regularization strength C and solver with GridSearchCV.

In [ ]:
# CO2: Logistic Regression with Feature Engineering and Regularization
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lr_grid = {
    "C": [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
    "solver": ["lbfgs", "liblinear"]
}

grid_lr = GridSearchCV(LogisticRegression(random_state=42, max_iter=1000), lr_grid, cv=skf, scoring="roc_auc", n_jobs=-1)
grid_lr.fit(X_train_scl, y_train)
best_lr = grid_lr.best_estimator_
print("Best LR Parameters:", grid_lr.best_params_)
print(f"Best CV ROC-AUC: {grid_lr.best_score_:.4f}")

## 6. Model 2: Random Forest Supervised Learning (CO3)
Tuning decision tree ensemble parameters.

In [ ]:
# CO3: Random Forest Tree-Based Supervised Learning
rf_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

grid_rf = GridSearchCV(RandomForestClassifier(random_state=42), rf_grid, cv=skf, scoring="roc_auc", n_jobs=-1)
grid_rf.fit(X_train_imp, y_train)
best_rf = grid_rf.best_estimator_
print("Best RF Parameters:", grid_rf.best_params_)
print(f"Best CV ROC-AUC: {grid_rf.best_score_:.4f}")

## 7. Model Evaluation & Comparison (CO5)
Holdout test set evaluation.

In [ ]:
# CO5: Stratified K-Fold Cross-Validation & Classification Metrics
def evaluate(model, X_test, y_test, name):
    yp = model.predict(X_test)
    yprob = model.predict_proba(X_test)[:, 1]
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, yp),
        "Precision": precision_score(y_test, yp),
        "Recall": recall_score(y_test, yp),
        "F1": f1_score(y_test, yp),
        "ROC-AUC": roc_auc_score(y_test, yprob)
    }

results = [
    evaluate(best_lr, X_test_scl, y_test, "Logistic Regression"),
    evaluate(best_rf, X_test_imp, y_test, "Random Forest")
]
pd.DataFrame(results).round(4)

## 8. Model Packaging (CO6)
Persisting models using joblib.

In [ ]:
# CO6: Model Packaging
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(best_lr, "../backend/models/logistic_regression.pkl")
joblib.dump(best_rf, "../backend/models/random_forest.pkl")
joblib.dump(scaler, "../backend/models/scaler.pkl")
joblib.dump(imputer, "../backend/models/imputer.pkl")
print("Models packaged to backend/models/")